<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/6AugRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
df  = pd.read_csv('/content/50_Startups - 50_Startups.csv')

In [10]:
df.head(10)

,R&D Spend,Administration,Marketing Spend,State,Profit
0,165349.20,136897.80,471784.10,New York,192261.83
1,162597.70,151377.59,443898.53,California,191792.06
2,153441.51,101145.55,407934.54,Florida,191050.39
3,144372.41,118671.85,383199.62,New York,182901.99
4,142107.34,91391.77,366168.42,Florida,166187.94
5,131876.90,99814.71,362861.36,New York,156991.12
6,134615.46,147198.87,127716.82,California,156122.51
7,130298.13,145530.06,323876.68,Florida,155752.60
8,120542.52,148718.95,311613.29,New York,152211.77
9,123334.88,108679.17,304981.62,California,149759.96


In [11]:
df.isnull().sum()

,0
R&D Spend,0
Administration,0
Marketing Spend,0
State,0
Profit,0


In [12]:
df.duplicated().sum()

np.int64(0)

In [25]:
X = df.iloc[: , :-1]
y = df.iloc[: , -1:]

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   R&D Spend        50 non-null     float64
 1   Administration   50 non-null     float64
 2   Marketing Spend  50 non-null     float64
 3   State            50 non-null     object 
 4   Profit           50 non-null     float64
dtypes: float64(4), object(1)
memory usage: 2.1+ KB


In [27]:
# apply one hot encoding on state column
X = pd.get_dummies(X, columns=['State'],drop_first=True , dtype=int)
X.head()

,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
0,165349.20,136897.80,471784.10,0,1
1,162597.70,151377.59,443898.53,0,0
2,153441.51,101145.55,407934.54,1,0
3,144372.41,118671.85,383199.62,0,1
4,142107.34,91391.77,366168.42,1,0


In [28]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

X_train , X_test ,y_train , y_test = train_test_split(X , y , test_size= 0.2, random_state=42)


In [29]:
model = LinearRegression()
model.fit(X_train , y_train)

LinearRegression()

In [31]:
print("regression intercept" , model.intercept_)
print("regression coefficient" , model.coef_)

regression intercept [54028.03959365]
regression coefficient [[ 8.05630064e-01 -6.87878823e-02  2.98554429e-02  9.38793006e+02
   6.98775997e+00]]


In [30]:
# check the mse and rmse
from sklearn.metrics import r2_score , mean_squared_error , root_mean_squared_error
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
rmse = np.sqrt(mse)
print("MSE: " , mse)
print("RMSE: " , rmse)

MSE:  82010363.04430099
RMSE:  9055.957323458464


In [32]:
# now build the mvr model using stats library
import statsmodels.api as sm
X_train = sm.add_constant(X_train)
X_test = sm.add_constant(X_test)
model = sm.OLS(y_train , X_train).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                     140.1
Date:                Thu, 06 Aug 2026   Prob (F-statistic):           1.13e-21
Time:                        05:18:44   Log-Likelihood:                -420.63
No. Observations:                  40   AIC:                             853.3
Df Residuals:                      34   BIC:                             863.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            5.403e+04   8457.293      6.388      0.000    3.68e+04    7.12e+04
R&D Spend           0.8056      0.052     15.383      0.000       0.699       0.912
Administration     -0.0688      0.061     -1.133      0.265      -0.192       0.055
Marketing Spend     0.0299      0.022      1.346      0.187      -0.015       0.075
State_Florida     938.7930   3893.511      0.241      0.811   -6973.773    8851.359
State_New York      6.9878   3882.765      0.002      0.999   -7883.740    7897.715
==============================================================================
Omnibus:                       15.391   Durbin-Watson:                   1.751
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               20.231
Skew:                          -1.142   Prob(JB):                     4.05e-05
Kurtosis:                       5.631   Cond. No.                     1.64e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.64e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [34]:
X_train.head()

,const,R&D Spend,Administration,Marketing Spend,State_Florida,State_New York
12,1.0,93863.75,127320.38,249839.44,1,0
4,1.0,142107.34,91391.77,366168.42,1,0
37,1.0,44069.95,51283.14,197029.42,0,0
8,1.0,120542.52,148718.95,311613.29,0,1
3,1.0,144372.41,118671.85,383199.62,0,1


In [35]:
# step1
X_opt = X_train
model = sm.OLS(y_train , X_opt).fit()
model.summary()


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.947
Method:                 Least Squares   F-statistic:                     140.1
Date:                Thu, 06 Aug 2026   Prob (F-statistic):           1.13e-21
Time:                        05:25:49   Log-Likelihood:                -420.63
No. Observations:                  40   AIC:                             853.3
Df Residuals:                      34   BIC:                             863.4
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            5.403e+04   8457.293      6.388      0.000    3.68e+04    7.12e+04
R&D Spend           0.8056      0.052     15.383      0.000       0.699       0.912
Administration     -0.0688      0.061     -1.133      0.265      -0.192       0.055
Marketing Spend     0.0299      0.022      1.346      0.187      -0.015       0.075
State_Florida     938.7930   3893.511      0.241      0.811   -6973.773    8851.359
State_New York      6.9878   3882.765      0.002      0.999   -7883.740    7897.715
==============================================================================
Omnibus:                       15.391   Durbin-Watson:                   1.751
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               20.231
Skew:                          -1.142   Prob(JB):                     4.05e-05
Kurtosis:                       5.631   Cond. No.                     1.64e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.64e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [40]:
X_opt = X_train.iloc[:,[0,1,2,3,4]]
# X_opt = X_train.drop(['State_New York'] , axis=1)
model = sm.OLS(y_train , X_opt).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.948
Method:                 Least Squares   F-statistic:                     180.2
Date:                Thu, 06 Aug 2026   Prob (F-statistic):           7.85e-23
Time:                        05:29:47   Log-Likelihood:                -420.63
No. Observations:                  40   AIC:                             851.3
Df Residuals:                      35   BIC:                             859.7
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            5.403e+04   8274.043      6.530      0.000    3.72e+04    7.08e+04
R&D Spend           0.8056      0.052     15.614      0.000       0.701       0.910
Administration     -0.0688      0.060     -1.150      0.258      -0.190       0.053
Marketing Spend     0.0299      0.022      1.374      0.178      -0.014       0.074
State_Florida     935.0793   3254.211      0.287      0.776   -5671.320    7541.479
==============================================================================
Omnibus:                       15.396   Durbin-Watson:                   1.751
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               20.244
Skew:                          -1.142   Prob(JB):                     4.02e-05
Kurtosis:                       5.633   Cond. No.                     1.63e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.63e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [41]:
X_opt = X_train.iloc[:,[0,1,2,3]]

# X_opt = X_train.drop(['State_Florida'] , axis=1)
model = sm.OLS(y_train , X_opt).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.954
Model:                            OLS   Adj. R-squared:                  0.950
Method:                 Least Squares   F-statistic:                     246.6
Date:                Thu, 06 Aug 2026   Prob (F-statistic):           4.76e-24
Time:                        05:30:11   Log-Likelihood:                -420.68
No. Observations:                  40   AIC:                             849.4
Df Residuals:                      36   BIC:                             856.1
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            5.407e+04   8166.660      6.621      0.000    3.75e+04    7.06e+04
R&D Spend           0.8038      0.051     15.906      0.000       0.701       0.906
Administration     -0.0679      0.059     -1.152      0.257      -0.188       0.052
Marketing Spend     0.0312      0.021      1.493      0.144      -0.011       0.074
==============================================================================
Omnibus:                       15.613   Durbin-Watson:                   1.759
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               21.146
Skew:                          -1.137   Prob(JB):                     2.56e-05
Kurtosis:                       5.742   Cond. No.                     1.63e+06
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.63e+06. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [42]:
X_opt = X_train.iloc[:,[0,1,3]]

# X_opt = X_train.drop(['State_Florida'] , axis=1)
model = sm.OLS(y_train , X_opt).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                 Profit   R-squared:                       0.952
Model:                            OLS   Adj. R-squared:                  0.949
Method:                 Least Squares   F-statistic:                     366.0
Date:                Thu, 06 Aug 2026   Prob (F-statistic):           4.19e-25
Time:                        05:32:32   Log-Likelihood:                -421.40
No. Observations:                  40   AIC:                             848.8
Df Residuals:                      37   BIC:                             853.9
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
===================================================================================
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const            4.554e+04   3457.277     13.173      0.000    3.85e+04    5.25e+04
R&D Spend           0.7834      0.048     16.480      0.000       0.687       0.880
Marketing Spend     0.0392      0.020      1.980      0.055      -0.001       0.079
==============================================================================
Omnibus:                       13.869   Durbin-Watson:                   1.753
Prob(Omnibus):                  0.001   Jarque-Bera (JB):               16.958
Skew:                          -1.067   Prob(JB):                     0.000208
Kurtosis:                       5.371   Cond. No.                     6.36e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 6.36e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""